# 10. Python Functions Basics (5+ Years Interview Guide)
Mastering function call stacks, parameter binding rules, the mutable default argument trap, variable unpacking (*args, **kwargs), and PEP 570 positional/keyword-only boundaries.

### Key 5-Year Interview Concepts Covered:
- **Mutable Default Argument Trap**: Why default parameters are evaluated once at function definition time in `__defaults__`.
- **Parameter Unpacking & Variadic Signatures**: `*args` (tuple) and `**kwargs` (dict) for dynamic wrapper layers.
- **Positional-Only (`/`) & Keyword-Only (`*`) Parameters**: Designing robust, future-proof public APIs.
- **Function Introspection & Type Annotations**: Inspecting `__code__`, `__annotations__`, and `inspect.signature`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Defining Functions & Call Stack Frames
**Explanation**: In Python, `def` is an executable statement that creates a function object (`PyFunctionObject`) and binds it to a name in the current namespace. When a function is called, CPython allocates a new execution frame on the call stack containing local variables (`LOAD_FAST`), bytecode evaluation offsets, and return addresses.

**Syntax**: `def function_name(param1, param2): return result`

In [ ]:
def fetch_system_status(): return 'Ok'
print(fetch_system_status())

### 2. Positional Arguments Mapping
**Explanation**: Positional arguments match parameters based on their order in the function signature. CPython copies passed argument references directly into the function frame's fast local array in O(1) time.

**Syntax**: `calculate_tax(amount, rate)`

In [ ]:
def compute_interest_rate(base, rate): return base * rate
print(compute_interest_rate(1000.0, 0.05))

### 3. Keyword Arguments Mapping
**Explanation**: Keyword arguments specify parameters by name `func(rate=0.05, amount=100.0)`. This allows passing arguments in any order, improves caller readability, and prevents accidental parameter inversion bugs in complex multi-argument functions.

**Syntax**: `func_name(param_name=value)`

In [ ]:
def compute_interest_rate(base, rate): return base * rate
print(compute_interest_rate(rate=0.05, base=1000.0))

### 4. Default Parameter Arguments
**Explanation**: Default parameter values allow callers to omit optional arguments. Python binds default values to the function object's `__defaults__` tuple when the `def` statement executes at module import time, not when the function is called.

**Syntax**: `def connect(host, port=5432, timeout=30): ...`

In [ ]:
def calculate_tax(amount, rate=0.18): return amount * rate
print(calculate_tax(100.0))

### 5. The Mutable Default Argument Pitfall
**Explanation**: Because default values are evaluated once at definition time, using a mutable object (e.g. `def append_to(item, target_list=[])`) means every invocation without that argument shares the EXACT same list instance stored in `func.__defaults__`! Appending to it mutates the shared default across subsequent calls.

**Syntax**: `def bad_fn(val, items=[]): items.append(val); return items  # Antipattern`

In [ ]:
def append_transaction_trap(tx_id, database_list=[]):
    database_list.append(tx_id)
    return database_list
print('c1:', append_transaction_trap('TX101'), '| c2:', append_transaction_trap('TX102'))

### 6. Safe None-Default Initialization Pattern
**Explanation**: The standard production solution to the mutable default trap is using `None` as the sentinel default and initializing a new instance inside the function body: `if target is None: target = []`. This guarantees a fresh instance is created on each invocation.

**Syntax**: `def good_fn(val, items=None): items = [] if items is None else items`

In [ ]:
def append_transaction_safe(tx_id, database_list=None):
    if database_list is None: database_list = []
    database_list.append(tx_id)
    return database_list
print('c1:', append_transaction_safe('TX101'), '| c2:', append_transaction_safe('TX102'))

### 7. Variable Positional Arguments (`*args`)
**Explanation**: The `*args` parameter collects extra positional arguments into an immutable tuple. It allows functions to accept an arbitrary number of inputs, and is the standard way decorator wrappers forward positional arguments.

**Syntax**: `def sum_all(*args): return sum(args)`

In [ ]:
def sum_all_payments(*payment_amounts):
    return sum(payment_amounts)
print(sum_all_payments(100.0, 200.0, 300.0))

### 8. Variable Keyword Arguments (`**kwargs`)
**Explanation**: The `**kwargs` parameter collects extra keyword arguments into a standard dictionary. It enables flexible dictionary forwarding, dynamic configuration overrides, and decorator wrappers that forward arbitrary keyword parameters.

**Syntax**: `def configure(**kwargs): for k, v in kwargs.items(): ...`

In [ ]:
def print_metadata_dict(**meta_dict):
    return meta_dict
print(print_metadata_dict(status='Completed', code=200))

### 9. Unpacking Sequence Arguments on Calls (`*list`)
**Explanation**: When calling a function, the `*` operator unpacks an iterable (list, tuple, set) into individual positional arguments: `func(*coordinates)`. This avoids manual indexing like `func(c[0], c[1])`.

**Syntax**: `func(*list_of_args)`

In [ ]:
def add_values(value_one, value_two): return value_one + value_two
values_list = [100.0, 200.0]
print(add_values(*values_list))

### 10. Unpacking Dictionary Arguments on Calls (`**dict`)
**Explanation**: The `**` operator unpacks a dictionary into keyword arguments on call: `create_user(**user_payload_dict)`. Keys in the dictionary must be strings matching parameter names in the function signature.

**Syntax**: `func(**kwargs_dict)`

In [ ]:
def calculate_net_amount(gross, tax): return gross - tax
argument_dict = {'tax': 20.0, 'gross': 100.0}
print(calculate_net_amount(**argument_dict))

### 11. Positional-Only Parameters (Python 3.8+ `/`)
**Explanation**: Parameters defined before a `/` slash in the signature are positional-only (PEP 570). Callers cannot pass them as keyword arguments (`func(x=1)` raises `TypeError`). This allows library authors to rename internal parameter names without breaking external callers and enables fast C-level argument parsing.

**Syntax**: `def format_id(id_val, /, prefix='TX'): ...`

In [ ]:
def print_payout(amount, /, currency='USD'): return f'${amount} {currency}'
print(print_payout(100.0, currency='EUR'))

### 12. Keyword-Only Parameters (`*`)
**Explanation**: Parameters defined after a bare `*` or `*args` are keyword-only. Callers MUST specify them by keyword name (e.g. `send_alert(msg, *, timeout=10)`). This prevents callers from accidentally passing confusing boolean flags as mystery positional arguments.

**Syntax**: `def query(sql, *, timeout=30, retry=True): ...`

In [ ]:
def print_payout(amount, *, currency): return f'${amount} {currency}'
print(print_payout(100.0, currency='EUR'))

### 13. Docstrings & Specification Standards
**Explanation**: Docstrings (`"""..."""`) placed immediately inside a function definition are stored in the function's `__doc__` attribute. Standard formats (Google style, Sphinx/NumPy style) document parameter types, return values, exceptions raised, and operational complexity for automated documentation generators.

**Syntax**: `def func(): """Short summary.\n\nArgs:...\nReturns:..."""`

In [ ]:
def run_audit():
    """Run transaction audit loops"""
print(run_audit.__doc__)

### 14. Type Annotations & Static Analysis
**Explanation**: Type hints (PEP 484) specify expected types for parameters and return values (`def process(amount: float) -> bool:`). Python does NOT enforce type hints at runtime; they are stored in `func.__annotations__` and used by static type checkers (mypy, pyright) and IDEs to catch bugs before production deployment.

**Syntax**: `def process_transaction(amount: float, status: str) -> bool: ...`

In [ ]:
def add_payouts(amount_one: float, amount_two: float) -> float:
    return amount_one + amount_two
print(add_payouts(5.5, 10.5))

### 15. Returning `None`: Explicit vs Implicit
**Explanation**: In Python, every function returns a value. If execution reaches the end of a function without hitting an explicit `return`, or hits a bare `return`, CPython implicitly returns the `None` singleton. In production code, explicitly write `return None` when returning `None` represents an intentional fallback outcome.

**Syntax**: `return None` / `return result_value`

In [ ]:
def empty_function(): pass
print(empty_function() is None)

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Designing robust transaction parsers using variadic parameter packing, keyword-only validation guards, and safe defaults.


In [ ]:
# Solution:
def dispatch(amount, /, *, target_wallet, logs=None):
    if logs is None: logs = []
    logs.append(amount)
    print(f'Sending ${amount:.2f} to {target_wallet}')
    return logs
    
with open(csv_path, 'r') as f:
    f.readline()
    row = f.readline().strip().split(',')
    amt = float(row[3]) if row[3] not in ('', 'NaN') else 0.0
    dispatch(amt, target_wallet='C82845_wallet')


### Q2: Dynamic Variadic Transaction Parser
**Explanation**: **Scenario**: Implement a transaction record parser using `*args` and `**kwargs` to accept variable transaction fields and apply dynamic metadata overrides.

**Syntax**: `def parse_tx(tx_id, *fields, **metadata): ...`

In [ ]:
# Solution:
def parse_metadata(*args, **kwargs):
    print('Tags:', args)
    print('Configurations:', kwargs)

parse_metadata('fintech', 'audit', mode='test', run_id=909)
